In [1]:
# GPU check
!nvidia-smi

# Clone Real-ESRGAN repository aur dependencies install karein
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
!pip install basicsr facexlib gfpgan
!pip install -r requirements.txt
!python setup.py develop

Mon Mar 16 16:17:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
import sys

# basicsr library ka installation path directly set karna
# Hum jante hain ki yeh /usr/local/lib/python3.12/dist-packages mein install hua hai
basicsr_install_path = None
for p in sys.path:
    if "dist-packages" in p and os.path.exists(os.path.join(p, "basicsr")):
        basicsr_install_path = os.path.join(p, "basicsr")
        break

if basicsr_install_path is None:
    # Fallback if the path isn't found in sys.path (unlikely after successful pip install)
    # Yeh path standard Google Colab environment ke liye hai
    basicsr_install_path = "/usr/local/lib/python3.12/dist-packages/basicsr"

deg_file = os.path.join(basicsr_install_path, 'data', 'degradations.py')

# File ko open kar ke galat import ko theek karna
if os.path.exists(deg_file):
    with open(deg_file, 'r') as f:
        content = f.read()

    # Sirf tab replace karein jab purana wala import string maujood ho
    old_import = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
    new_import = 'from torchvision.transforms.functional import rgb_to_grayscale'

    if old_import in content:
        content = content.replace(old_import, new_import)

        # Theek kiya hua code wapis save karna
        with open(deg_file, 'w') as f:
            f.write(content)
        print("✅ basicsr degradations.py file updated successfully!")
    else:
        print("ℹ️ basicsr degradations.py file mein pehle se hi sahi import hai ya badlav ki zaroorat nahi hai.")
else:
    print(f"❌ Error: degradations.py file '{deg_file}' nahi mila.")

# Ab basicsr ko import karne ki koshish karein taaki verify ho sake ki fix kaam kar gaya.
try:
    import basicsr
    print("✅ basicsr successfully imported after fix.")
    print("Ab aap apna upscaling wala code dobara run kar sakte hain.")
except ModuleNotFoundError as e:
    print(f"❌ basicsr import abhi bhi fail ho raha hai: {e}")

✅ basicsr degradations.py file updated successfully!
✅ basicsr successfully imported after fix.
Ab aap apna upscaling wala code dobara run kar sakte hain.


In [3]:
!pip install gradio

In [3]:
import os
import cv2
import gradio as gr
import subprocess

# Output folder setup
output_dir = "/content/output/"
os.makedirs(output_dir, exist_ok=True)

def process_image(image_path, resolution, model, tile_size):
    print("Processing started...")

    # Auto-Convert to PNG to preserve alpha/colors
    img_temp = cv2.imread(image_path)
    if img_temp is None:
        return None

    temp_input_path = "/content/temp_input.png"
    cv2.imwrite(temp_input_path, img_temp)

    # Original dimensions
    orig_h, orig_w = img_temp.shape[:2]

    # Scale factor mapping
    scale_factor = 4
    if "2 x" in resolution:
        scale_factor = 2
    elif "3 x" in resolution:
        scale_factor = 3

    # Run Real-ESRGAN via subprocess
    command = f"python /content/Real-ESRGAN/inference_realesrgan.py -n {model} -i {temp_input_path} -o {output_dir} -s {scale_factor} --ext png --tile {tile_size}"
    os.system(command)

    # Expected output path
    out_path = os.path.join(output_dir, "temp_input_out.png")

    # Resize logic
    if os.path.exists(out_path):
        img_out = cv2.imread(out_path)
        if img_out is not None:
            if "Same as Original" in resolution:
                resized_img = cv2.resize(img_out, (orig_w, orig_h), interpolation=cv2.INTER_AREA)
                cv2.imwrite(out_path, resized_img)
            elif "original" not in resolution:
                res_str = resolution.split("(")[1].split(")")[0]
                target_w, target_h = map(int, res_str.split(" x "))
                resized_img = cv2.resize(img_out, (target_w, target_h), interpolation=cv2.INTER_CUBIC)
                cv2.imwrite(out_path, resized_img)

        return out_path
    else:
        return None

# Gradio Interface Design
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("## 2D Anime & Image Upscaler")
    gr.Markdown("Upload an image, select your model and resolution, and enhance your art instantly.")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="filepath", label="Upload Image")

            model_dropdown = gr.Dropdown(
                choices=["RealESRGAN_x4plus", "RealESRNet_x4plus", "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"],
                value="RealESRGAN_x4plus_anime_6B",
                label="Select AI Model"
            )

            resolution_dropdown = gr.Dropdown(
                choices=["Same as Original (Enhance Only)", "FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)", "8k (7680 x 4320)", "2 x original", "3 x original", "4 x original"],
                value="Same as Original (Enhance Only)",
                label="Target Resolution"
            )

            tile_slider = gr.Slider(minimum=128, maximum=1024, step=128, value=512, label="Tile Size (Lower to avoid Memory Error)")

            submit_btn = gr.Button("Upscale Image", variant="primary")

        with gr.Column():
            output_img = gr.Image(type="filepath", label="Upscaled Result")

    submit_btn.click(
        fn=process_image,
        inputs=[input_img, resolution_dropdown, model_dropdown, tile_slider],
        outputs=output_img
    )

# Launch with share=True to get a public link
app.launch(share=True, debug=True, allowed_paths=["/content/output/"])

/tmp/ipykernel_6564/324854116.py:56: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c84f4c5143a990a938.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 